# Home Credit — Default Prediction Model (lean notebook)

The essential modeling pipeline only, extracted from `home_credit_eda.ipynb` (which remains the
full record: EDA Parts 1–7 and modeling experiments 8.4–8.5, 8.8–8.11, 8.13).

Contents: **Setup → 8.1** build all 95 engineered features + Logistic Regression → **8.2** choose the
operating threshold → **8.3** dashboard → **8.6** LightGBM → **8.7** model comparison → **8.12** one-hot
encoding (best model, AUC ≈ 0.780) → **8.14** precision-first operating points → **8.15** LR learning curves.

Run top to bottom (~5 minutes). The CSVs must sit in the same folder as the notebook.

In [ ]:
# Setup — imports, data folder, plot style
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Folder that contains the CSV files. Default = current working directory.
DATA_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
# If auto-detection is wrong, hard-code it, e.g.:
# DATA_DIR = r'C:\Users\Amr\Downloads\home-credit-default-risk'

def path(name):
    return os.path.join(DATA_DIR, name)

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'axes.axisbelow': True,
    'font.size': 11,
})
BLUE   = '#4E79A7'   # neutral / non-default
ORANGE = '#F28E2B'   # default / emphasis
GREY   = '#BAB0AC'   # reference lines

app_train = pd.read_csv(path('application_train.csv'))
print('application_train:', app_train.shape)

---
# Part 8 — Risk-oriented Logistic Regression (recall-first)

**Goal:** predict who will default using **all the engineered features** — application-level *plus* every auxiliary table — tuned so that **false negatives (missed defaulters) are nearly zero**, while keeping false positives as low as that allows.

The steps:
1. **Documents, all 20 types** — the provided-% table by loan type shows the document pattern is largely product-driven (DOC_3: 78% of cash vs 3.9% of revolving clients), so `*_unexpected` anomaly flags and `× revolving` interactions are built for every document with real variance.
2. **Feature table in memory** — 95 engineered features: 68 application-level (ratios, EXT aggregates + missing flags, cleaned employment, documents) **+ 27 from the six auxiliary tables** using the same aggregations as Part 6 (bureau debt/overdue/microloans, worst-ever & chronic DPD, prior approval/refusal & upsell ratio, installment lateness & underpayment, POS delinquency, credit-card utilization & over-limit share), plus `has_*` history flags so median imputation can't disguise "no history" as "typical client". No intermediate files.
3. **Logistic Regression** — median-impute and StandardScaler **fit on the train split only**, `class_weight='balanced'` for the 92/8 imbalance.
4. **Threshold choice** — 8.2 first shows the recall-first extreme (what catching ~every defaulter costs), then a menu of *usable* operating points and settles on one via the single `OPERATING_THRESHOLD` variable, defaulting to **0.50**.
5. **Report + visuals** — classification report, confusion matrix, ROC, precision–recall, threshold sweep, and coefficient chart, all at the chosen operating point.

**The honest tradeoff:** AUC ≈ 0.756 is fixed by the model; the threshold just picks where on the curve you sit. Chasing near-zero false negatives means flagging almost everyone (99% recall ⇒ ~93% flagged) — useless in practice. The default **0.50** operating point catches **~68%** of defaulters while flagging only **~33%** of applicants (repaid-class recall 0.70, default-class precision 16.5%) — a usable model. Move `OPERATING_THRESHOLD` down to catch more defaulters, up to flag fewer people; the 8.2 menu shows the exact numbers at each point.

In [ ]:
## 8.1 Build features (in memory) and train the Logistic Regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, classification_report, confusion_matrix

if 'app_train' not in globals():
    app_train = pd.read_csv(path('application_train.csv'))

d = app_train
doc_cols = [c for c in d.columns if c.startswith('FLAG_DOCUMENT_')]
cash = d['NAME_CONTRACT_TYPE'] == 'Cash loans'
rev  = d['NAME_CONTRACT_TYPE'] == 'Revolving loans'

# --- STEP 1: provided % by loan type, ALL 20 documents ---
tbl = pd.DataFrame({
    'cash_%':      [d.loc[cash, c].mean()*100 for c in doc_cols],
    'revolving_%': [d.loc[rev,  c].mean()*100 for c in doc_cols],
    'overall_%':   [d[c].mean()*100 for c in doc_cols]}, index=doc_cols).round(2)
print('Documents by loan type (top 8 by overall %):')
print(tbl.sort_values('overall_%', ascending=False).head(8).to_string())
VARYING = ['FLAG_DOCUMENT_3','FLAG_DOCUMENT_6','FLAG_DOCUMENT_8','FLAG_DOCUMENT_5','FLAG_DOCUMENT_16','FLAG_DOCUMENT_18']

# --- STEP 2: application-level features ---
F = d[['SK_ID_CURR']].copy()
inc  = d['AMT_INCOME_TOTAL'].replace(0, np.nan)
cred = d['AMT_CREDIT'].replace(0, np.nan)
F['credit_income_ratio']  = d['AMT_CREDIT'] / inc
F['annuity_income_ratio'] = d['AMT_ANNUITY'] / inc
F['credit_goods_ratio']   = d['AMT_CREDIT'] / d['AMT_GOODS_PRICE'].replace(0, np.nan)
F['payment_rate']         = d['AMT_ANNUITY'] / cred
F['credit_minus_goods']   = d['AMT_CREDIT'] - d['AMT_GOODS_PRICE']
F['income_per_person']    = inc / d['CNT_FAM_MEMBERS'].replace(0, np.nan)
F['age_years'] = -d['DAYS_BIRTH'] / 365.25
emp_anom = d['DAYS_EMPLOYED'] == 365243
F['days_employed_anomaly'] = emp_anom.astype(int)
F['days_employed_clean'] = d['DAYS_EMPLOYED'].where(~emp_anom, np.nan)
F['employed_to_age_ratio'] = F['days_employed_clean'] / d['DAYS_BIRTH']
F['days_last_phone_change'] = d['DAYS_LAST_PHONE_CHANGE']
F['days_id_publish'] = d['DAYS_ID_PUBLISH']
F['days_registration'] = d['DAYS_REGISTRATION']
ext = d[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']]
F['EXT_SOURCE_1'] = d['EXT_SOURCE_1']; F['EXT_SOURCE_2'] = d['EXT_SOURCE_2']; F['EXT_SOURCE_3'] = d['EXT_SOURCE_3']
F['EXT_mean'] = ext.mean(axis=1); F['EXT_min'] = ext.min(axis=1); F['EXT_max'] = ext.max(axis=1)
F['n_ext_present'] = ext.notna().sum(axis=1)
for c in ext.columns: F[c + '_missing'] = d[c].isna().astype(int)
bldg_kw = ['APARTMENTS','AREA','YEARS','FLOOR','ELEVATORS','ENTRANCES','LANDAREA','LIVINGAREA','NONLIVING','COMMONAREA','BASEMENT','TOTALAREA']
bcols = [c for c in d.columns if any(k in c for k in bldg_kw)]
F['n_building_cols_present'] = d[bcols].notna().sum(axis=1)
enq = ['AMT_REQ_CREDIT_BUREAU_HOUR','AMT_REQ_CREDIT_BUREAU_DAY','AMT_REQ_CREDIT_BUREAU_WEEK','AMT_REQ_CREDIT_BUREAU_MON','AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR']
F['enquiry_block_missing'] = d[enq].isna().all(axis=1).astype(int)
F['total_enquiries'] = d[enq].sum(axis=1)
F['is_revolving'] = rev.astype(int)
F['flag_own_car'] = (d['FLAG_OWN_CAR'] == 'Y').astype(int)
F['flag_own_realty'] = (d['FLAG_OWN_REALTY'] == 'Y').astype(int)
F['own_car_age'] = d['OWN_CAR_AGE'].fillna(-1)
F['cnt_children'] = d['CNT_CHILDREN']
F['region_rating'] = d['REGION_RATING_CLIENT']
F['n_address_mismatch'] = d[['REG_CITY_NOT_LIVE_CITY','REG_CITY_NOT_WORK_CITY','LIVE_CITY_NOT_WORK_CITY']].sum(axis=1)
F['social_default_ratio'] = np.where(d['OBS_30_CNT_SOCIAL_CIRCLE'] > 0,
    d['DEF_30_CNT_SOCIAL_CIRCLE'] / d['OBS_30_CNT_SOCIAL_CIRCLE'], np.nan)
F['occupation_median_income'] = d.groupby('OCCUPATION_TYPE')['AMT_INCOME_TOTAL'].transform('median')
F['n_documents'] = d[doc_cols].sum(axis=1)
for c in doc_cols: F[c.lower()] = d[c]
for c in VARYING:                                     # anomaly-vs-norm + explicit LR interactions
    norm_cash = d.loc[cash, c].mean() >= 0.5
    norm_rev  = d.loc[rev,  c].mean() >= 0.5
    F[c.lower() + '_unexpected'] = np.where(cash, d[c] != int(norm_cash), d[c] != int(norm_rev)).astype(int)
    F[c.lower() + '_x_revolving'] = rev.astype(int) * d[c]
n_app_feats = F.shape[1] - 1
print(f'\nApplication-level features: {n_app_feats}')

# --- STEP 2b: auxiliary-table features (same aggregations as Part 6) ---
# bureau — external debt load, microloans, overdue history, recency
bureau = pd.read_csv(path('bureau.csv'),
    usecols=['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_TYPE',
             'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_OVERDUE', 'AMT_CREDIT_MAX_OVERDUE', 'DAYS_CREDIT'])
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    n_bureau_credits=('SK_ID_BUREAU', 'count'),
    total_debt=('AMT_CREDIT_SUM_DEBT', 'sum'),
    total_overdue=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    n_active=('CREDIT_ACTIVE', lambda s: (s == 'Active').sum()),
    has_microloan=('CREDIT_TYPE', lambda s: (s == 'Microloan').any()),
    max_overdue_ever=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    most_recent_days_credit=('DAYS_CREDIT', 'max'),   # max = closest to 0 = most recent
)
bureau_agg['recency_years'] = -bureau_agg['most_recent_days_credit'] / 365.0
bureau_agg['has_microloan'] = bureau_agg['has_microloan'].astype(int)
F = F.merge(bureau_agg.drop(columns=['most_recent_days_credit']), on='SK_ID_CURR', how='left')
for c in ['n_bureau_credits', 'total_debt', 'total_overdue', 'n_active', 'has_microloan']:
    F[c] = F[c].fillna(0)   # no bureau rows = a real zero, not missing
credit_by_id = d.set_index('SK_ID_CURR')['AMT_CREDIT']
F['debt_to_current_credit'] = F['total_debt'] / F['SK_ID_CURR'].map(credit_by_id).replace(0, np.nan)

# bureau_balance — worst-ever & chronic DPD (rows reach the client via SK_ID_BUREAU)
bb = pd.read_csv(path('bureau_balance.csv'), usecols=['SK_ID_BUREAU', 'STATUS'])
dpd_map = {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, 'C': 0}   # 'X' (unknown) -> NaN, ignored by max()
bb['DPD'] = bb['STATUS'].map(dpd_map)
bb['is_dpd'] = bb['STATUS'].isin(['1', '2', '3', '4', '5'])
bb_known = bb[bb['STATUS'] != 'X']
loan_worst = bb.groupby('SK_ID_BUREAU')['DPD'].max()
loan_share = bb_known.groupby('SK_ID_BUREAU').agg(dpd_months=('is_dpd', 'sum'), obs_months=('is_dpd', 'size'))
bur_key = bureau[['SK_ID_CURR', 'SK_ID_BUREAU']].copy()
bur_key['worst_dpd'] = bur_key['SK_ID_BUREAU'].map(loan_worst)
bur_key = bur_key.join(loan_share, on='SK_ID_BUREAU')
client_bb = bur_key.groupby('SK_ID_CURR').agg(
    worst_ever_dpd=('worst_dpd', 'max'),
    dpd_months_sum=('dpd_months', 'sum'),
    obs_months_sum=('obs_months', 'sum'),
)
client_bb['chronic_dpd_share'] = client_bb['dpd_months_sum'] / client_bb['obs_months_sum'].replace(0, np.nan)
F = F.merge(client_bb[['worst_ever_dpd', 'chronic_dpd_share']], on='SK_ID_CURR', how='left')
# obs_months_sum > 0 = at least one observed (non-'X') month — clients with bureau rows but no
# bureau_balance coverage must get 0 here, or the flag would just repeat n_bureau_credits > 0
F['has_bureau_balance'] = F['SK_ID_CURR'].isin(client_bb.index[client_bb['obs_months_sum'] > 0]).astype(int)
del bb, bb_known, bur_key, bureau   # free the 27M-row table before loading the next one

# previous applications — approval rate, refusals, upsell ratio, top yield tier
prev = pd.read_csv(path('previous_application.csv'),
    usecols=['SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'AMT_APPLICATION', 'AMT_CREDIT', 'NAME_YIELD_GROUP'])
pg = prev.groupby('SK_ID_CURR')['NAME_CONTRACT_STATUS']
F['prev_approval_rate'] = F['SK_ID_CURR'].map(pg.apply(lambda s: (s == 'Approved').mean()))
F['ever_refused'] = (F['SK_ID_CURR'].map(pg.apply(lambda s: (s == 'Refused').any())) == True).astype(int)
F['n_prev_applications'] = F['SK_ID_CURR'].map(prev.groupby('SK_ID_CURR').size()).fillna(0)
ap = prev[(prev.NAME_CONTRACT_STATUS == 'Approved') & (prev.AMT_APPLICATION > 0)].copy()
ap['ratio'] = ap['AMT_CREDIT'] / ap['AMT_APPLICATION']
F['credit_haircut_ratio'] = F['SK_ID_CURR'].map(ap.groupby('SK_ID_CURR')['ratio'].mean())
yorder = {'low_action': 1, 'low_normal': 2, 'middle': 3, 'high': 4}
py = prev[prev.NAME_YIELD_GROUP.isin(yorder)].copy()
py['yld'] = py['NAME_YIELD_GROUP'].map(yorder)
F['top_yield_group'] = F['SK_ID_CURR'].map(py.groupby('SK_ID_CURR')['yld'].max())

# installments — lateness & underpayment
inst = pd.read_csv(path('installments_payments.csv'),
    usecols=['SK_ID_CURR', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT'])
inst['days_late'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['underpaid'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
late_agg = inst.groupby('SK_ID_CURR').agg(
    avg_days_late=('days_late', 'mean'),
    max_days_late=('days_late', 'max'),
    avg_underpaid=('underpaid', 'mean'),
)
F = F.merge(late_agg, on='SK_ID_CURR', how='left')
F['has_installment_history'] = F['SK_ID_CURR'].isin(late_agg.index).astype(int)
del inst

# POS / cash — tolerance-adjusted delinquency
pos = pd.read_csv(path('POS_CASH_balance.csv'), usecols=['SK_ID_CURR', 'SK_DPD_DEF'])
pos['late'] = (pos['SK_DPD_DEF'] > 0).astype(int)
pos_agg = pos.groupby('SK_ID_CURR').agg(pos_max_dpd_def=('SK_DPD_DEF', 'max'), pos_late_share=('late', 'mean'))
F = F.merge(pos_agg, on='SK_ID_CURR', how='left')
F['has_pos_history'] = F['SK_ID_CURR'].isin(pos_agg.index).astype(int)

# credit card — utilization & over-limit share
cc = pd.read_csv(path('credit_card_balance.csv'),
    usecols=['SK_ID_CURR', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'SK_DPD_DEF'])
cc['util'] = cc['AMT_BALANCE'] / cc['AMT_CREDIT_LIMIT_ACTUAL'].replace(0, np.nan)
cc_lim = cc[cc['AMT_CREDIT_LIMIT_ACTUAL'] > 0].copy()   # exclude dormant/closed cards (limit=0)
cc_lim['over'] = (cc_lim['AMT_BALANCE'] > cc_lim['AMT_CREDIT_LIMIT_ACTUAL']).astype(int)
cc_agg = cc.groupby('SK_ID_CURR').agg(cc_avg_utilization=('util', 'mean'), cc_max_dpd_def=('SK_DPD_DEF', 'max'))
F = F.merge(cc_agg, on='SK_ID_CURR', how='left')
F = F.merge(cc_lim.groupby('SK_ID_CURR')['over'].mean().rename('cc_over_limit_share'), on='SK_ID_CURR', how='left')
F['has_cc_history'] = F['SK_ID_CURR'].isin(cc_agg.index).astype(int)
# NOTE on the has_* flags: median-imputing e.g. avg_days_late for a client with no installment
# history would silently label them a "typical payer" — the flag lets the model see the difference.

y = d['TARGET']
X = F.drop(columns=['SK_ID_CURR'])
print(f'Feature table: {X.shape[1]} features ({n_app_feats} application-level + {X.shape[1] - n_app_feats} from auxiliary tables), {len(X):,} rows')

# --- STEP 3: train (impute + scale fit on train split ONLY) ---
Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
med = Xtr.median()
sc = StandardScaler().fit(Xtr.fillna(med))
lr = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)
lr.fit(sc.transform(Xtr.fillna(med)), ytr)
p = lr.predict_proba(sc.transform(Xv.fillna(med)))[:, 1]
auc = roc_auc_score(yv, p)
print(f'Validation AUC: {auc:.4f}')

In [ ]:
## 8.2 Threshold sweep + choose a usable operating point
from sklearn.metrics import f1_score
fpr, tpr, thr = roc_curve(yv, p)
n_pos, n_neg = int(yv.sum()), int((1 - yv).sum())
print(f'Validation set: {n_pos:,} defaulters / {n_neg:,} good clients\n')

# (a) The recall-first extreme: what catching almost every defaulter costs.
rows = []
for target_rec in [1.00, 0.99, 0.95, 0.90, 0.80]:
    ix = np.where(tpr >= target_rec)[0][0]
    t = thr[ix]
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yv, pred).ravel()
    rows.append({'recall_target': f'{target_rec:.0%}', 'threshold': round(float(t), 4),
                 'caught(TP)': tp, 'missed(FN)': fn, 'false_pos(FP)': fp,
                 'FP_rate': f'{fp/n_neg:.1%}', 'flagged_%': f'{(tp+fp)/len(yv):.1%}',
                 'precision': f'{tp/(tp+fp):.1%}'})
print('(a) Recall-first sweep — chasing high recall flags almost everyone:')
print(pd.DataFrame(rows).to_string(index=False))

# (b) Usable operating points: raise the threshold so the model flags a sensible
#     minority instead of ~everyone. F1-optimal = the cutoff that best balances
#     precision and recall on the default class.
grid = np.linspace(0.05, 0.95, 181)
t_f1 = float(grid[int(np.argmax([f1_score(yv, (p >= t).astype(int)) for t in grid]))])
menu = []
for name, t in [('default 0.50', 0.50), ('balanced 0.60', 0.60),
                (f'F1-optimal {t_f1:.2f}', round(t_f1, 3)), ('conservative 0.70', 0.70)]:
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yv, pred).ravel()
    rec, prec = tp / (tp + fn), tp / (tp + fp)
    menu.append({'operating_point': name, 'threshold': t, 'flagged_%': f'{(tp+fp)/len(yv):.1%}',
                 'recall(def)': f'{rec:.1%}', 'precision(def)': f'{prec:.1%}',
                 'F1(def)': round(2 * prec * rec / (prec + rec), 3), 'missed(FN)': fn, 'false_pos(FP)': fp})
print('\n(b) Usable operating points — the model no longer flags almost everyone:')
print(pd.DataFrame(menu).to_string(index=False))

# --- CHOSEN OPERATING POINT: change this one number to move along the trade-off above ---
# 0.50 = the standard cutoff: catches ~68% of defaulters while flagging ~33% of applicants
# (not 93%). Lower it toward 0.35-0.40 to catch more defaulters; raise it to flag fewer people.
OPERATING_THRESHOLD = 0.50
pred_op = (p >= OPERATING_THRESHOLD).astype(int)
cm = confusion_matrix(yv, pred_op)
tn_op, fp_op, fn_op, tp_op = cm.ravel()
tpr_op, fpr_op = tp_op / n_pos, fp_op / n_neg    # ROC marker coordinates for 8.3
print(f'\nClassification report @ operating threshold {OPERATING_THRESHOLD:.2f} '
      f'(flags {(tp_op+fp_op)/len(yv):.1%} of applicants, catches {tpr_op:.1%} of defaulters):')
print(classification_report(yv, pred_op, target_names=['repaid(0)', 'default(1)'], digits=3))

In [ ]:
## 8.3 Visualize — ROC, precision-recall, threshold sweep, confusion matrix
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

ax = axes[0, 0]                                                   # ROC
ax.plot(fpr, tpr, color=BLUE, lw=2, label=f'LR (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], color=GREY, ls=':', label='chance')
ax.scatter([fpr_op], [tpr_op], color=ORANGE, zorder=5, label=f'operating point (thr {OPERATING_THRESHOLD:.2f})')
ax.set_xlabel('False positive rate'); ax.set_ylabel('Recall'); ax.set_title('ROC curve'); ax.legend()

ax = axes[0, 1]                                                   # precision-recall
prec, rec, _ = precision_recall_curve(yv, p)
ax.plot(rec, prec, color=BLUE, lw=2)
ax.axhline(yv.mean(), color=GREY, ls=':', label=f'baseline ({yv.mean():.1%})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('Precision-Recall curve'); ax.legend()

ax = axes[1, 0]                                                   # threshold sweep
ax.plot(thr[1:], tpr[1:], color=BLUE, lw=2, label='recall (catch rate)')
ax.plot(thr[1:], fpr[1:], color=ORANGE, lw=2, label='false positive rate')
ax.axvline(OPERATING_THRESHOLD, color=GREY, ls='--', label=f'operating thr ({OPERATING_THRESHOLD:.2f})')
ax.set_xlabel('Decision threshold'); ax.set_ylabel('Rate'); ax.set_xlim(0, 1)
ax.set_title('Threshold sweep'); ax.legend()

ax = axes[1, 1]                                                   # confusion matrix
im = ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f'{v:,}', ha='center', va='center',
            color='white' if v > cm.max()/2 else 'black', fontsize=13)
ax.set_xticks([0, 1], ['pred: repaid', 'pred: default'])
ax.set_yticks([0, 1], ['true: repaid', 'true: default'])
ax.set_title(f'Confusion matrix @ threshold {OPERATING_THRESHOLD:.2f}'); ax.grid(False)

plt.tight_layout(); plt.show()

## 8.6 LightGBM — gradient-boosted trees on the same features

Same 95 features, same train/validation split, same `class_weight='balanced'` — but a gradient-boosted tree model instead of a linear one. Run this after 8.1 and 8.2. Two things are deliberately different from the LR pipeline:

1. **No imputation, no scaling.** LightGBM handles missing values natively (each split learns which branch NaNs go to — "missingness as signal" for free) and is invariant to feature scale, so it trains on the raw feature table. The `has_*` flags and `*_missing` columns stay in — trees can still use them.
2. **Early stopping on an *inner* split of the training data** (10% carved off `Xtr`), never on the validation set — otherwise the number of trees would be tuned on the same rows we report AUC on, and the comparison with LR would be unfairly optimistic.

Colab has LightGBM preinstalled; the guarded import below auto-installs it anywhere else. Expect a clear AUC gain over the logistic regression — trees capture non-linearities (e.g. risk vs age isn't a straight line) and feature interactions the linear model can't.

In [ ]:
## 8.6 LightGBM on the same 95 features — no imputation, no scaling needed
try:
    import lightgbm as lgb
except ImportError:                       # Colab has it preinstalled; this covers local runs
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb

# inner split for early stopping — the real validation set (Xv) stays untouched
X_fit, X_es, y_fit, y_es = train_test_split(Xtr, ytr, test_size=0.1, random_state=42, stratify=ytr)

lgbm = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.05, num_leaves=31,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    min_child_samples=100, reg_alpha=0.1, reg_lambda=0.1,
    metric='auc',   # otherwise LightGBM ALSO tracks binary_logloss and early-stops on whichever stalls first
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
         callbacks=[lgb.early_stopping(100, verbose=False, first_metric_only=True)])
print(f'Trees kept by early stopping: {lgbm.best_iteration_} of 2000')

p_lgb = lgbm.predict_proba(Xv)[:, 1]      # predict_proba uses the early-stopped iteration
auc_lgb = roc_auc_score(yv, p_lgb)
print(f'Validation AUC — LogisticRegression: {auc:.4f}  |  LightGBM: {auc_lgb:.4f}')

# same usable-operating-points menu as 8.2, now for LightGBM probabilities
menu = []
for name, t in [('default 0.50', 0.50), ('balanced 0.60', 0.60), ('conservative 0.70', 0.70)]:
    pred = (p_lgb >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yv, pred).ravel()
    rec, prec = tp / (tp + fn), tp / (tp + fp)
    menu.append({'operating_point': name, 'threshold': t, 'flagged_%': f'{(tp+fp)/len(yv):.1%}',
                 'recall(def)': f'{rec:.1%}', 'precision(def)': f'{prec:.1%}',
                 'F1(def)': round(2 * prec * rec / (prec + rec), 3), 'missed(FN)': fn, 'false_pos(FP)': fp})
print('\nUsable operating points (LightGBM):')
print(pd.DataFrame(menu).to_string(index=False))

pred_lgb = (p_lgb >= OPERATING_THRESHOLD).astype(int)
tp_l = int(((pred_lgb == 1) & (yv == 1)).sum()); fl_l = int((pred_lgb == 1).sum())
print(f'\nClassification report @ operating threshold {OPERATING_THRESHOLD:.2f} '
      f'(flags {fl_l/len(yv):.1%} of applicants, catches {tp_l/yv.sum():.1%} of defaulters):')
print(classification_report(yv, pred_lgb, target_names=['repaid(0)', 'default(1)'], digits=3))

# --- visuals: ROC overlay vs the LR + what the trees actually split on ---
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))

ax = axes[0]
fpr_l, tpr_l, _ = roc_curve(yv, p_lgb)
ax.plot(fpr_l, tpr_l, color=ORANGE, lw=2, label=f'LightGBM (AUC={auc_lgb:.3f})')
ax.plot(fpr, tpr, color=BLUE, lw=2, label=f'LogisticRegression (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], color=GREY, ls=':', label='chance')
ax.set_xlabel('False positive rate'); ax.set_ylabel('Recall')
ax.set_title('ROC — LightGBM vs Logistic Regression'); ax.legend()

ax = axes[1]
imp = pd.Series(lgbm.booster_.feature_importance(importance_type='gain'),
                index=X.columns).sort_values().tail(20)
ax.barh(imp.index, imp.values, color=ORANGE)
ax.set_xlabel('total gain (loss reduction across all splits)')
ax.set_title('LightGBM: top 20 features by gain')

plt.tight_layout(); plt.show()

## 8.7 Model comparison — Logistic Regression vs LightGBM

Both models were trained on the **identical 95 features**, the **identical train/validation split**, and the same `class_weight='balanced'` — so this is a clean head-to-head, not an apples-to-oranges one. Three ways to read the comparison below:

- **AUC** — threshold-independent ranking quality. Higher = better at separating defaulters from repayers across *every* cutoff at once.
- **Metrics at the shared operating threshold (0.50)** — what each model actually does at the operating point chosen in 8.2.
- **Matched recall** — hold the catch-rate fixed (same fraction of defaulters caught) and ask which model flags fewer good clients to get there. This is the fairest view, because the two models' probability scales aren't directly comparable.

**How to choose — in lending it isn't only about AUC:**

| | Logistic Regression | LightGBM |
|---|---|---|
| Predictive power (AUC) | 0.756 | **0.777** |
| Explainability | **Signed coefficients** — each feature's direction & magnitude, ready for adverse-action reasons | Gain importances (magnitude only, no direction); needs SHAP for per-applicant explanations |
| Preprocessing | Needs median-impute + scaling | Handles raw NaNs, scale-invariant |
| Non-linearities & interactions | Must be hand-engineered | Learned automatically |

Regulators (e.g. ECOA/FCRA in the US) require you to explain *why* an applicant was declined, and a logistic regression's signed coefficients give that essentially for free. LightGBM's ~0.02 AUC edge is real but modest; if you deploy it you'll typically pair it with SHAP to recover per-applicant reasons. A common production answer is **LightGBM for the score, LR as the interpretable challenger/baseline** — which is exactly what this notebook now has.

In [ ]:
## 8.7 Model comparison — Logistic Regression vs LightGBM
def cm_metrics(probs, t):
    tn, fp, fn, tp = confusion_matrix(yv, (probs >= t).astype(int)).ravel()
    rec, prec = tp / (tp + fn), tp / (tp + fp)
    return dict(flagged=(tp + fp) / len(yv), recall=rec, precision=prec,
                f1=2 * prec * rec / (prec + rec), fn=fn, fp=fp)

models = [('LogisticRegression', p, auc), ('LightGBM', p_lgb, auc_lgb)]

# (1) AUC (threshold-independent) + what each model does at the shared operating threshold
disp = []
for name, probs, a in models:
    m = cm_metrics(probs, OPERATING_THRESHOLD)
    disp.append({'model': name, 'AUC': round(a, 4), 'flagged_%': f"{m['flagged']:.1%}",
                 'recall(def)': f"{m['recall']:.1%}", 'precision(def)': f"{m['precision']:.1%}",
                 'F1(def)': round(m['f1'], 3), 'missed(FN)': m['fn'], 'false_pos(FP)': m['fp']})
print(f'(1) Head-to-head @ operating threshold {OPERATING_THRESHOLD:.2f}:')
print(pd.DataFrame(disp).to_string(index=False))
print(f"\n    AUC gain from LightGBM: +{auc_lgb - auc:.4f}  ({(auc_lgb - auc) / auc * 100:+.1f}% relative)")

# (2) matched catch-rate: hold recall fixed, compare the false-positive cost
rows2 = []
for target_rec in [0.90, 0.80, 0.68]:
    for name, probs, a in models:
        fpr_, tpr_, thr_ = roc_curve(yv, probs)
        t = float(thr_[np.where(tpr_ >= target_rec)[0][0]])
        m = cm_metrics(probs, t)
        rows2.append({'recall_target': f'{target_rec:.0%}', 'model': name, 'threshold': round(t, 3),
                      'false_pos(FP)': m['fp'], 'flagged_%': f"{m['flagged']:.1%}",
                      'precision(def)': f"{m['precision']:.1%}"})
print('\n(2) At matched recall (same defaulters caught) — which flags fewer good clients?')
print(pd.DataFrame(rows2).to_string(index=False))

# (3) visual — default-class metrics at the operating threshold, side by side
lr_m, lg_m = cm_metrics(p, OPERATING_THRESHOLD), cm_metrics(p_lgb, OPERATING_THRESHOLD)
labels = ['recall', 'precision', 'F1']
lr_vals = [lr_m['recall'], lr_m['precision'], lr_m['f1']]
lg_vals = [lg_m['recall'], lg_m['precision'], lg_m['f1']]
xpos = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(7.5, 4.5))
b1 = ax.bar(xpos - w / 2, lr_vals, w, color=BLUE, label=f'LogisticRegression (AUC {auc:.3f})')
b2 = ax.bar(xpos + w / 2, lg_vals, w, color=ORANGE, label=f'LightGBM (AUC {auc_lgb:.3f})')
ax.set_xticks(xpos, labels); ax.set_ylabel('score'); ax.set_ylim(0, 1)
ax.set_title(f'Default-class metrics @ threshold {OPERATING_THRESHOLD:.2f}')
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=9); ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=9)
ax.legend()
plt.tight_layout(); plt.show()

## 8.12 One-hot encoding — does adding the categorical columns change the results?

Six categorical columns have been absent from the model so far: `NAME_EDUCATION_TYPE`, `NAME_INCOME_TYPE`, `NAME_FAMILY_STATUS`, `NAME_HOUSING_TYPE`, `OCCUPATION_TYPE` (so far only summarized as one median-income number) and `ORGANIZATION_TYPE`. Here each category becomes its own 0/1 column (**one-hot encoding**, ~100 new columns), both models retrain on the extended table, and we compare against the 95-feature baselines to see what actually changes. Run after 8.1, 8.2 and 8.6.

Why one-hot: a 0/1 column per category adds no fake ordering, so it's the honest encoding for a linear model — the logistic regression finally gets to see, e.g., the education gradient (lower-secondary defaults ≈ 5× academic-degree). LightGBM can consume the same dummies, though for trees they mostly spread existing information thinner.

Everything stays leakage-safe: the encoding is structural (no `TARGET` involved), the split reuses the exact same rows as 8.1, and imputation/scaling statistics are still fit on the train split only. `CODE_GENDER` is deliberately left out: sex is a protected attribute under fair-lending rules (ECOA), so the credit model shouldn't consume it even though the dataset provides it.

*Production note:* for LightGBM specifically there's a usually-better third route — native categorical support (`astype('category')`; it learns optimal category groupings directly, no dummies needed).

In [ ]:
## 8.12 One-hot encode the 6 excluded categorical columns and retrain both models
import re

CAT_COLS = ['NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS',
            'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']
for c in CAT_COLS:
    print(f'{c:>22}: {d[c].nunique():>2} categories, {d[c].isna().mean():.1%} missing')

# one 0/1 column per category; rows with NaN (e.g. OCCUPATION_TYPE) get all-zeros, which
# IS the missing-indicator. Sanitize names: LightGBM rejects special JSON characters.
dummies = pd.get_dummies(d[CAT_COLS], dtype=int)
dummies.columns = [re.sub(r'[^0-9A-Za-z_]+', '_', c) for c in dummies.columns]
X_oh = pd.concat([X, dummies], axis=1)
print(f'\nFeature table: {X.shape[1]} -> {X_oh.shape[1]} features (+{dummies.shape[1]} one-hot columns)')

# same validation rows as 8.1 (reuse the split indices), preprocessing still train-only
Xtr_oh, Xv_oh = X_oh.loc[Xtr.index], X_oh.loc[Xv.index]

# --- Logistic Regression on the one-hot table ---
med_oh = Xtr_oh.median()
sc_oh = StandardScaler().fit(Xtr_oh.fillna(med_oh))
lr_oh = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)
lr_oh.fit(sc_oh.transform(Xtr_oh.fillna(med_oh)), ytr)
p_lr_oh = lr_oh.predict_proba(sc_oh.transform(Xv_oh.fillna(med_oh)))[:, 1]
auc_lr_oh = roc_auc_score(yv, p_lr_oh)

# --- LightGBM on the one-hot table (same pipeline as 8.6) ---
Xf_oh, Xes_oh, yf_oh, yes_oh = train_test_split(Xtr_oh, ytr, test_size=0.1, random_state=42, stratify=ytr)
lgbm_oh = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.05, num_leaves=31,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    min_child_samples=100, reg_alpha=0.1, reg_lambda=0.1,
    metric='auc', class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
lgbm_oh.fit(Xf_oh, yf_oh, eval_set=[(Xes_oh, yes_oh)], eval_metric='auc',
            callbacks=[lgb.early_stopping(100, verbose=False, first_metric_only=True)])
p_lgb_oh = lgbm_oh.predict_proba(Xv_oh)[:, 1]
auc_lgb_oh = roc_auc_score(yv, p_lgb_oh)

# --- did the results change? ---
cmp = pd.DataFrame([
    {'model': 'LogisticRegression', 'baseline_AUC': round(auc, 4),
     'one-hot_AUC': round(auc_lr_oh, 4), 'delta': f'{auc_lr_oh - auc:+.4f}'},
    {'model': 'LightGBM', 'baseline_AUC': round(auc_lgb, 4),
     'one-hot_AUC': round(auc_lgb_oh, 4), 'delta': f'{auc_lgb_oh - auc_lgb:+.4f}'},
])
print('\nDid one-hot encoding change the results?')
print(cmp.to_string(index=False))

# strongest new dummies as the LR sees them (signed, standardized)
oh_coefs = pd.Series(lr_oh.coef_[0], index=X_oh.columns)[dummies.columns]
top_new = oh_coefs.reindex(oh_coefs.abs().sort_values(ascending=False).index).head(8)
print('\nStrongest new one-hot features (LR coefficient, + pushes toward default):')
print(top_new.round(3).to_string())

for probs, a, name in [(p_lr_oh, auc_lr_oh, 'LogisticRegression + one-hot'),
                       (p_lgb_oh, auc_lgb_oh, 'LightGBM + one-hot')]:
    pred = (probs >= OPERATING_THRESHOLD).astype(int)
    tp_ = int(((pred == 1) & (yv == 1)).sum()); fl_ = int((pred == 1).sum())
    print(f'\n{name}  (AUC = {a:.4f})')
    print(f'Classification report @ operating threshold {OPERATING_THRESHOLD:.2f} '
          f'(flags {fl_/len(yv):.1%} of applicants, catches {tp_/yv.sum():.1%} of defaulters):')
    print(classification_report(yv, pred, target_names=['repaid(0)', 'default(1)'], digits=3))

## 8.14 Enhancing precision — precision-first operating points

Precision on the default class has two honest levers: **raise the threshold** (flag fewer, surer
people — precision rises, recall falls) or **use a better-ranking model** (more recall at the same
precision, for free). For each precision target below, the threshold shown keeps the **maximum
possible recall**, for the full-feature Logistic Regression and the best model (LightGBM + one-hot).

Ceiling to remember: only ~8% of applicants default, so there is no threshold giving high precision
*and* high recall at AUC ≈ 0.76–0.78 — closing that gap needs a better model, not a different cutoff.

In [ ]:
## 8.14 Precision-first operating points — max recall at each precision target
def precision_menu(probs, name, targets=(0.15, 0.20, 0.25, 0.30, 0.40)):
    prec_c, rec_c, thr_c = precision_recall_curve(yv, probs)
    rows = []
    for tgt in targets:
        ok = np.where(prec_c[:-1] >= tgt)[0]          # thresholds reaching the target precision
        if len(ok) == 0:
            rows.append({'precision_target': f'{tgt:.0%}', 'threshold': '—',
                         'recall(def)': 'unreachable', 'flagged_%': '—',
                         'caught(TP)': '—', 'missed(FN)': '—', 'false_pos(FP)': '—'})
            continue
        i = ok[np.argmax(rec_c[:-1][ok])]             # among them, keep the most recall
        t = float(thr_c[i])
        pred = (probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(yv, pred).ravel()
        rows.append({'precision_target': f'{tgt:.0%}', 'threshold': round(t, 3),
                     'recall(def)': f'{tp/(tp+fn):.1%}', 'flagged_%': f'{(tp+fp)/len(yv):.1%}',
                     'caught(TP)': tp, 'missed(FN)': fn, 'false_pos(FP)': fp})
    print(f'\n{name}:')
    print(pd.DataFrame(rows).to_string(index=False))

print('What each precision target costs in recall (threshold picked for max recall at that precision):')
precision_menu(p,   f'Logistic Regression (full features) (AUC {auc:.3f})')
precision_menu(p_lgb_oh, f'LightGBM + one-hot — best model (AUC {auc_lgb_oh:.3f})')

# enhanced-precision report: the best model at the 25%-precision operating point
prec_c, rec_c, thr_c = precision_recall_curve(yv, p_lgb_oh)
ok = np.where(prec_c[:-1] >= 0.25)[0]
t25 = float(thr_c[ok[np.argmax(rec_c[:-1][ok])]])
pred25 = (p_lgb_oh >= t25).astype(int)
tp_ = int(((pred25 == 1) & (yv == 1)).sum()); fl_ = int((pred25 == 1).sum())
print(f'\nEnhanced-precision report — LightGBM + one-hot @ threshold {t25:.3f} '
      f'(flags {fl_/len(yv):.1%}, catches {tp_/yv.sum():.1%}):')
print(classification_report(yv, pred25, target_names=['repaid(0)', 'default(1)'], digits=3))

# visual: both PR curves with the 25%-precision operating point marked
fig, ax = plt.subplots(figsize=(7.5, 5.5))
pl, rl, _ = precision_recall_curve(yv, p)
ax.plot(rl, pl, color=BLUE, lw=2, label=f'LR (full features) (AUC {auc:.3f})')
ax.plot(rec_c, prec_c, color=ORANGE, lw=2, label=f'LightGBM + one-hot (AUC {auc_lgb_oh:.3f})')
ax.scatter([tp_ / yv.sum()], [tp_ / fl_], color='#B22222', zorder=5, label=f'25%-precision point (thr {t25:.2f})')
ax.axhline(yv.mean(), color=GREY, ls=':', label=f'baseline ({yv.mean():.1%})')
ax.set_xlabel('Recall (defaulters caught)'); ax.set_ylabel('Precision (flags that are real)')
ax.set_title('Precision-recall: better model = more recall at every precision'); ax.legend()
plt.tight_layout(); plt.show()

## 8.15 Logistic Regression learning curves — training size & regularization

The LR analogue of 8.13. A linear model has no trees to add, so its two learning curves are:

1. **AUC vs training-set size** — refit the identical pipeline on growing stratified subsamples of the training split, always scoring on the same untouched validation set. If the validation curve is still rising at full size, more data would help (underfit by data); if it plateaued long ago and the train–validation gap is small, the model has extracted what a linear form can and more rows won't help.
2. **AUC vs `C` (inverse regularization strength)** — sklearn's LR applies L2 regularization controlled by `C` (default 1.0; *smaller* C = *stronger* regularization). Too small → coefficients crushed toward zero, both curves low (underfit). Too large → weak protection; with 95 features and 246k rows expect the train AUC to creep up while validation stays flat or dips (mild overfit).

All preprocessing (median impute + scaler) is refit inside each run on that run's training subsample only — the validation rows never leak into any statistic. Run after 8.1. Takes a few minutes: it fits the logistic regression ~10 times.

In [ ]:
## 8.15 LR learning curves — AUC vs training size, and AUC vs regularization (C)
def lr_train_valid_auc(Xt, yt, C=1.0):
    """Fit the identical LR pipeline on (Xt, yt); return (train AUC, validation AUC)."""
    m = Xt.median()
    s = StandardScaler().fit(Xt.fillna(m))
    model = LogisticRegression(C=C, max_iter=3000, class_weight='balanced', random_state=42)
    model.fit(s.transform(Xt.fillna(m)), yt)
    a_tr = roc_auc_score(yt, model.predict_proba(s.transform(Xt.fillna(m)))[:, 1])
    a_va = roc_auc_score(yv, model.predict_proba(s.transform(Xv.fillna(m)))[:, 1])
    return a_tr, a_va

# (a) learning curve vs training-set size (stratified subsamples of the training split)
sizes = [10_000, 25_000, 50_000, 100_000, len(Xtr)]
size_tr, size_va = [], []
print('AUC vs training-set size:')
for n in sizes:
    if n >= len(Xtr):
        Xs, ys = Xtr, ytr
    else:
        Xs, _, ys, _ = train_test_split(Xtr, ytr, train_size=n, random_state=42, stratify=ytr)
    a_tr, a_va = lr_train_valid_auc(Xs, ys)
    size_tr.append(a_tr); size_va.append(a_va)
    print(f'  n={len(Xs):>7,}: train AUC {a_tr:.4f} | validation AUC {a_va:.4f} | gap {a_tr - a_va:+.4f}')

# (b) validation curve vs C (inverse L2 regularization strength; sklearn default C=1)
Cs = [0.001, 0.01, 0.1, 1.0, 10.0]
c_tr, c_va = [], []
print('\nAUC vs regularization strength C (smaller C = stronger regularization):')
for Cv in Cs:
    a_tr, a_va = lr_train_valid_auc(Xtr, ytr, C=Cv)
    c_tr.append(a_tr); c_va.append(a_va)
    print(f'  C={Cv:<6}: train AUC {a_tr:.4f} | validation AUC {a_va:.4f} | gap {a_tr - a_va:+.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

ax = axes[0]
ax.plot(sizes, size_tr, color=BLUE, lw=2, marker='o', label='train AUC')
ax.plot(sizes, size_va, color=ORANGE, lw=2, marker='o', label='validation AUC')
ax.set_xlabel('training-set size (rows)'); ax.set_ylabel('AUC')
ax.set_title('Learning curve — AUC vs training size'); ax.legend(loc='lower right')
ax.ticklabel_format(axis='x', style='plain')

ax = axes[1]
ax.plot(Cs, c_tr, color=BLUE, lw=2, marker='o', label='train AUC')
ax.plot(Cs, c_va, color=ORANGE, lw=2, marker='o', label='validation AUC')
ax.axvline(1.0, color=GREY, ls='--', label='sklearn default (C=1)')
ax.set_xscale('log')
# same y-span as the left panel — otherwise autoscale zooms into a 0.0002-wide range
# and pure sampling noise masquerades as meaningful curves; the honest story is "flat"
ax.set_ylim(0.740, 0.765)
ax.set_xlabel('C  (inverse regularization strength, log scale)'); ax.set_ylabel('AUC')
ax.set_title('Validation curve — AUC vs regularization (flat = C has no effect)'); ax.legend(loc='lower right')

plt.tight_layout(); plt.show()

## 8.16 SHAP values — from "which features matter" to "why this applicant"

Gain importance (8.6) says which features the model's accuracy came from, but it has no sign and no per-applicant story. **SHAP values** fix both: for every single prediction, the applicant's score is split fairly across the features, so each feature gets a signed contribution (+ pushes toward default, − away), and the contributions **add up exactly** to that applicant's log-odds score. Run after 8.1 and 8.6.

Implementation note: no `shap` library needed — LightGBM computes exact TreeSHAP natively via `booster_.predict(..., pred_contrib=True)`. The cell verifies the additivity property (base value + contributions = raw score) before plotting.

Reading the summary (beeswarm) panel, which shows the **top 10 features by gain**: each dot is one validation applicant; x-position is that applicant's SHAP value for the feature; color is the applicant's feature value (red = high, blue = low, grey = missing). `EXT_mean` reading "blue dots on the right, red on the left" means *low* external scores push *toward* default — the direction gain alone couldn't tell you. The right panel explains one concrete applicant (the riskiest in the validation set), the exact per-person breakdown a loan officer or regulator would ask for.

In [ ]:
## 8.16 SHAP values (exact TreeSHAP, built into LightGBM — no extra library)
contrib = lgbm.booster_.predict(Xv, pred_contrib=True)   # (rows, 96): 95 features + base value
shap_vals = pd.DataFrame(contrib[:, :-1], columns=X.columns)
base_value = contrib[0, -1]
raw = lgbm.booster_.predict(Xv, raw_score=True)
assert np.allclose(base_value + shap_vals.sum(axis=1), raw, atol=1e-4)   # additivity check
print(f'SHAP matrix: {shap_vals.shape[0]:,} applicants x {shap_vals.shape[1]} features | '
      f'base value (average log-odds): {base_value:+.3f}')

# global view: the top 10 features BY GAIN, with their SHAP importance alongside
gain = pd.Series(lgbm.booster_.feature_importance(importance_type='gain'), index=X.columns)
mean_abs = shap_vals.abs().mean()
top_gain = gain.sort_values(ascending=False).head(10).index.tolist()
cmpr = pd.DataFrame({'gain_rank': gain.rank(ascending=False).astype(int)[top_gain],
                     'SHAP_rank': mean_abs.rank(ascending=False).astype(int)[top_gain],
                     'mean_|SHAP|': mean_abs[top_gain].round(4)})
print('\nTop 10 features by gain — and how SHAP ranks the same features:')
print(cmpr.to_string())

rng = np.random.default_rng(42)
sample = rng.choice(len(Xv), size=4000, replace=False)   # 4k dots per feature keeps the plot legible


def fmt(v):
    return 'NaN' if pd.isna(v) else f'{v:,.3g}'


fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1.4, 1]})

ax = axes[0]                                              # beeswarm-style SHAP summary
for i, f in enumerate(reversed(top_gain)):
    sv = shap_vals[f].to_numpy()[sample]
    fv = Xv[f].to_numpy(dtype=float)[sample]
    ys = i + rng.uniform(-0.28, 0.28, len(sv))
    miss = np.isnan(fv)
    pct = pd.Series(fv).rank(pct=True).to_numpy()         # color = percentile of the feature value
    ax.scatter(sv[~miss], ys[~miss], c=pct[~miss], cmap='coolwarm', s=4, alpha=0.6)
    ax.scatter(sv[miss], ys[miss], color=GREY, s=4, alpha=0.5)
ax.axvline(0, color=GREY, lw=1)
ax.set_yticks(range(len(top_gain)), list(reversed(top_gain)))
ax.set_xlabel('SHAP value (log-odds; > 0 pushes toward default)')
ax.set_title('SHAP summary — top 10 features by gain\n(red = high value, blue = low, grey = missing)')

ax = axes[1]                                              # one applicant, fully explained
i_risk = int(np.argmax(raw))
row = shap_vals.iloc[i_risk]
top8 = row.reindex(row.abs().sort_values(ascending=False).index).head(8)[::-1]
labels = [f'{f} = {fmt(Xv.iloc[i_risk][f])}' for f in top8.index]
ax.barh(labels, top8.values, color=[ORANGE if v > 0 else BLUE for v in top8.values])
ax.axvline(0, color=GREY, lw=1)
ax.set_xlabel('SHAP value (log-odds)')
p_risk = 1 / (1 + np.exp(-raw[i_risk]))
ax.set_title(f'Riskiest validation applicant (p = {p_risk:.2f}):\nthe 8 largest contributions to that score')
plt.tight_layout(); plt.show()

print(f'\nRiskiest applicant: predicted default probability {p_risk:.1%} '
      f'(true label: {"default" if yv.iloc[i_risk] == 1 else "repaid"})')